# RL basics

Термины и понятия:

- агент/среда
- наблюдение $o$ / состояние $s$
- действие $a$, стратегия $\pi: \pi(s) \rightarrow a$ функция перехода $T: T(s, a) \rightarrow s'$
- вознаграждение $r$, ф-я вознаграждений $R: R(s, a) \rightarrow r$
- цикл взаимодействия, траектория $\tau: (s_0, a_0, r_0, s_1, a_1, r_1, ..., s_T, a_T, r_T)$, эпизод
- отдача $G$, подсчет отдачи, средняя[/ожидаемая] отдача $\mathbb{E}[G]$

In [1]:
try:
    import google.colab
    COLAB = True
except ModuleNotFoundError:
    COLAB = False
    pass

if COLAB:
    !pip -q install "gymnasium[classic-control, atari, accept-rom-license]"
    !pip -q install piglet
    !pip -q install imageio_ffmpeg
    !pip -q install moviepy==1.0.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 11.8 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 41.0 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle-environments 1.18.0 requires shimmy>=1.2.1, but you have shimmy 0.2.1 which is incompatible.
dopamine-rl 4.1.2 requires ale-py>=0.10.1, but you have ale-py 0.8.1 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.5/67.5 kB 2.0 MB/s eta 0:00:00


In [2]:
import glob
import io
import base64
import gymnasium as gym
import numpy as np
from IPython import display as ipythondisplay
from IPython.display import HTML
import matplotlib.pyplot as plt
%matplotlib inline

## Agent, environment

<img src=https://gymnasium.farama.org/_images/lunar_lander.gif caption="lunar lander" width="150" height="50"><img src=https://gymnasium.farama.org/_images/mountain_car.gif caption="mountain car" width="150" height="50">
<img src=https://gymnasium.farama.org/_images/cliff_walking.gif caption="cliff walking" width="300" height="50">
<img src=https://ale.farama.org/_images/montezuma_revenge.gif caption="montezuma revenge" width="150" height="100">
<img src=https://github.com/danijar/crafter/raw/main/media/video.gif caption="crafter" width="150" height="100">
<img src=https://camo.githubusercontent.com/6df2ca438d8fe8aa7a132b859315147818c54af608f8609320c3c20e938acf48/68747470733a2f2f6d656469612e67697068792e636f6d2f6d656469612f344e78376759694d394e44724d724d616f372f67697068792e676966 caption="malmo minecraft" width="150" height="100">
<img src=https://images.ctfassets.net/kftzwdyauwt9/e0c0947f-1a44-4528-4a41450a9f0a/2d0e85871d58d02dbe01b2469d693d4a/table-03.gif caption="roboschool" width="150" height="100">
<img src=https://raw.githubusercontent.com/Tviskaron/mipt/master/2019/RL/02/mdp.png caption="Марковский процесс принятия решений" width="150" height="100">
<img src=https://minigrid.farama.org/_images/DoorKeyEnv.gif caption="minigrid" width="120" height="120">

## Observation, state

TODO:
- добавить примеры наблюдений/состояний (числа, векторы, картинки)
- интуитивное объяснение различия, положить пока, что наблюдение = состояние
- пространство состояний


В каждый момент времени среда имеет некоторое внутреннее состояние. Здесь слово "состояние" я употребил скорее в интуитивном понимании, чтобы обозначить, что среда изменчива (иначе какой смысл с ней взаимодействовать, если ничего не меняется). В обучении с подкреплением под термином состояние $s$ (или $s_t$, где $t$ — текущее время) подразумевают либо абстрактно информацию о "состоянии" среды, либо ее явное представление в виде данных, достаточные для полного описания "состояния". *NB: Здесь можно провести аналогию с компьютерными играми — файл сохранения игры как раз содержит информацию о "состоянии" мира игры, чтобы в будущем можно было продолжить с текущей точки, так что данные этого файла в целом можно с некоторой натяжкой считать состоянием (с натяжкой, потому что редко когда в сложных играх файлы сохранения содержат прямо вот всю информацию, так что после перезагрузки вы получите не совсем точную копию). При этом обычно подразумевается, что состояние не содержит в себе ничего лишнего, то есть это **минимальный** набор информации.*

Наблюдением $o$ называют то, что агент "видит" о текущем состоянии среды. Это не обязательно зрение, а вообще вся доступная ему информация (условно, со всех его органов чувств).

В общем случае наблюдение: кортеж/словарь многомерных векторов чисел.

In [3]:
print(gym.make("CartPole-v0").reset()[0].shape)
print(gym.make("MountainCar-v0").reset()[0].shape)

(4,)
(2,)


/usr/local/lib/python3.11/dist-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


## Action, policy, transition function

Рассмотрим следующие MDP:

- A: <img src=https://i.ibb.co/mrCMVZLQ/mdp-a.png caption="A" width="400" height="100">

Links to all:
[A](https://i.ibb.co/mrCMVZLQ/mdp-a.png)
[B](https://i.ibb.co/GQ2tVtjC/mdp-b.png)
[C](https://i.ibb.co/Jj9LYHjP/mdp-c.png)
[D](https://i.ibb.co/Y47Mr83b/mdp-d.png)
[E](https://i.ibb.co/Kjt1Xhmf/mdp-e.png)

Давайте явно запишем пространства состояний $S$ и действий $A$, а также функцию перехода $T$ среды.

In [4]:
states = set(range(3))
actions = set(range(1))

print(f'{states=} | {actions=}')

T = {
    (0, 0): 1,
    (1, 0): 2,
    (2, 0): 2
}
print(f'Transition function {T=}')

A_mdp = states, actions, T

states={0, 1, 2} | actions={0}
Transition function T={(0, 0): 1, (1, 0): 2, (2, 0): 2}


Попробуйте записать функцию перехода в матричном виде:

In [5]:
T_matrix = np.array([
    [0, 1, 0],
    [0, 0, 1],
    [0, 0, 1]
])

Как получить вероятность нахождения агента в состоянии (1) через N шагов? Что происходит с вероятностями нахождения в состояниях при $N \rightarrow \infty$

In [6]:
# 1. нужно вектор начального состояния умножить на (матрицу перехода)^N 
#    и посмотреть на соответствующий элемент получившегося вектора
# 2. мы притопаем 100%-но в 2ое состояние, т.к. на каждом шаге переходим в следующее состояние, 
#    если еще не во 2ом. Для наглядности покажем что это так даже для состояния 0.

p_0 = np.array([1, 0, 0])

p_1 = p_0 @ T_matrix
print(f"Распределение вероятностей через 1 шаг: {p_1}")
print(f"Вероятность быть в состоянии 1 через 1 шаг: {p_1[1]}\n")

p_100 = p_0 @ np.linalg.matrix_power(T_matrix, 100)
print(f"Распределение вероятностей через много шагов: {p_100}")

Распределение вероятностей через 1 шаг: [0 1 0]
Вероятность быть в состоянии 1 через 1 шаг: 1

Распределение вероятностей через много шагов: [0 0 1]


Задайте еще несколько MDP:

- B: <img src=https://i.ibb.co/GQ2tVtjC/mdp-b.png caption="B" width="400" height="100">

- C: <img src=https://i.ibb.co/Jj9LYHjP/mdp-c.png caption="C" width="400" height="100">

In [7]:
states_b = set(range(4))
actions_b = set(range(3))

print(f'{states_b=} | {actions_b=}')

T_b = {
    (0, 0): 1, (0, 1): 2, (0, 2): 3,
    (1, 0): 1, (1, 1): 1, (1, 2): 1,
    (2, 0): 2, (2, 1): 2, (2, 2): 2,
    (3, 0): 3, (3, 1): 3, (3, 2): 3
}
print(f'Transition function {T_b=}')

B_mdp = states_b, actions_b, T_b

T_matrix_B = np.array([    # uniform
    [0, 1/3, 1/3, 1/3],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
])

states_b={0, 1, 2, 3} | actions_b={0, 1, 2}
Transition function T_b={(0, 0): 1, (0, 1): 2, (0, 2): 3, (1, 0): 1, (1, 1): 1, (1, 2): 1, (2, 0): 2, (2, 1): 2, (2, 2): 2, (3, 0): 3, (3, 1): 3, (3, 2): 3}


In [8]:
states_c = set(range(4))
actions_c = set(range(2))

print(f'{states_c=} | {actions_c=}')

T_c = {
    (0, 0): 1, (0, 1): 2,
    (1, 0): 1, (1, 1): 3,
    (2, 0): 3, (2, 1): 2,
    (3, 0): 3, (3, 1): 3
}
print(f'Transition function {T_c=}')

C_mdp = states_c, actions_c, T_c

T_matrix_C = np.array([  # uniform
    [0.0, 1/2, 1/2, 0.0],
    [0.0, 1/2, 0.0, 1/2],
    [0.0, 0.0, 1/2, 1/2],
    [0.0, 0.0, 0.0, 1.0]
])

states_c={0, 1, 2, 3} | actions_c={0, 1}
Transition function T_c={(0, 0): 1, (0, 1): 2, (1, 0): 1, (1, 1): 3, (2, 0): 3, (2, 1): 2, (3, 0): 3, (3, 1): 3}


Давайте попробуем задать двух агентов: случайного и оптимального (для каждой среды свой).

In [9]:
class Agent:
    def __init__(self, actions):
        self.rng = np.random.default_rng()
        self.actions = np.array(list(actions))

    def act(self, state):
        return self.rng.integers(len(self.actions))


class OptimalAgent:
    def __init__(self, policy: dict):
        self.policy = policy

    def act(self, state):
        return self.policy[state]

В качестве дополнения, запишите стратегию агента

In [10]:
pi_A = {0: 0, 1: 0, 2: 0}
pi_B = {0: 2}
pi_C = {0: 0, 1: 1, 2: 0, 3: 0}

## Reward, reward function

Теперь добавим произвольную функцию вознаграждения. Например, для A:

In [11]:
R = {
    (0, 0): -0.1,
    (1, 0): 1.0,
    (2, 0): 0.0
}
print(R)

A_mdp = *A_mdp, R
print(A_mdp)

{(0, 0): -0.1, (1, 0): 1.0, (2, 0): 0.0}
({0, 1, 2}, {0}, {(0, 0): 1, (1, 0): 2, (2, 0): 2}, {(0, 0): -0.1, (1, 0): 1.0, (2, 0): 0.0})


In [12]:
R_b = {
    (0, 0): 0.1, (0, 1): 0.5, (0, 2): 1.0,
    (1, 0): 0.0, (1, 1): 0.0, (1, 2): 0.0,
    (2, 0): 0.0, (2, 1): 0.0, (2, 2): 0.0,
    (3, 0): 0.0, (3, 1): 0.0, (3, 2): 0.0
}
print(R_b)

B_mdp = *B_mdp, R_b
print(B_mdp)

{(0, 0): 0.1, (0, 1): 0.5, (0, 2): 1.0, (1, 0): 0.0, (1, 1): 0.0, (1, 2): 0.0, (2, 0): 0.0, (2, 1): 0.0, (2, 2): 0.0, (3, 0): 0.0, (3, 1): 0.0, (3, 2): 0.0}
({0, 1, 2, 3}, {0, 1, 2}, {(0, 0): 1, (0, 1): 2, (0, 2): 3, (1, 0): 1, (1, 1): 1, (1, 2): 1, (2, 0): 2, (2, 1): 2, (2, 2): 2, (3, 0): 3, (3, 1): 3, (3, 2): 3}, {(0, 0): 0.1, (0, 1): 0.5, (0, 2): 1.0, (1, 0): 0.0, (1, 1): 0.0, (1, 2): 0.0, (2, 0): 0.0, (2, 1): 0.0, (2, 2): 0.0, (3, 0): 0.0, (3, 1): 0.0, (3, 2): 0.0})


In [13]:
R_c = {
    (0, 0): -0.1, (0, 1): -0.1,
    (1, 0): -0.1, (2, 1): -0.1,
    (1, 1): 1.0, (2, 0): 1.0, 
    (3, 0): 0.0, (3, 1): 0.0
}
print(R_c)

C_mdp = *C_mdp, R_c
print(C_mdp)

{(0, 0): -0.1, (0, 1): -0.1, (1, 0): -0.1, (2, 1): -0.1, (1, 1): 1.0, (2, 0): 1.0, (3, 0): 0.0, (3, 1): 0.0}
({0, 1, 2, 3}, {0, 1}, {(0, 0): 1, (0, 1): 2, (1, 0): 1, (1, 1): 3, (2, 0): 3, (2, 1): 2, (3, 0): 3, (3, 1): 3}, {(0, 0): -0.1, (0, 1): -0.1, (1, 0): -0.1, (2, 1): -0.1, (1, 1): 1.0, (2, 0): 1.0, (3, 0): 0.0, (3, 1): 0.0})


## Interaction loop, trajectory, termination, truncation, episode

Общий цикл взаимодействия в рамках эпизода:
1. Инициализировать среду: $s \leftarrow \text{env.init()}$
2. Цикл:
    - выбрать действие: $a \leftarrow \pi(s)$
    - получить ответ от среды: $s, r, d \leftarrow \text{env.next(a)}$
    - если $d == \text{True}$, выйти из цикла

In [14]:
def run_episode(mdp):
    states, actions, T, R = mdp
    agent = Agent(actions)

    s = 0
    tau = []
    for _ in range(5):
        a = agent.act(s)
        s_next = T[(s, a)]
        r = R[(s, a)]

        tau.append((s, a, r))
        s = s_next

    return tau

run_episode(A_mdp)

[(0, 0, -0.1), (1, 0, 1.0), (2, 0, 0.0), (2, 0, 0.0), (2, 0, 0.0)]

Termination — означает окончание эпизода, когда достигнуто терминальное состояние. Является частью задания среды.

Truncation — означает окончание эпизода, когда достигнут лимит по числу шагов (=времени). Обычно является внешне заданным параметром для удобства обучения.

Пока не будем вводить truncation, но поддержим termination: расширьте определение среды информацией о терминальных состояниях для всех описанных ранее сред. Сгенерируйте по несколько случайных траекторий для каждой среды.

In [15]:
terminals_A = {2}
terminals_B = {1, 2, 3}
terminals_C = {3}

A_mdp = *A_mdp, terminals_A
B_mdp = *B_mdp, terminals_B
C_mdp = *C_mdp, terminals_C

random_agent_A = Agent(A_mdp[1])
random_agent_B = Agent(B_mdp[1])
random_agent_C = Agent(C_mdp[1])

def run_episode_term(mdp, agent):
    states, actions, T, R, terminals = mdp

    s = 0
    tau = []
    for _ in range(5):
        a = agent.act(s)
        s_next = T[(s, a)]
        r = R[(s, a)]

        tau.append((s, a, r))
        s = s_next
        if s in terminals:
            break

    return tau

for i in range(3):
    trajectory_A = run_episode_term(A_mdp, random_agent_A)
    trajectory_B = run_episode_term(B_mdp, random_agent_B)
    trajectory_C = run_episode_term(C_mdp, random_agent_C)
    print(f"{i+1}-ая случайная траектория для А: {trajectory_A}")
    print(f"{i+1}-ая случайная траектория для B: {trajectory_B}")
    print(f"{i+1}-ая случайная траектория для C: {trajectory_C}")

1-ая случайная траектория для А: [(0, 0, -0.1), (1, 0, 1.0)]
1-ая случайная траектория для B: [(0, 2, 1.0)]
1-ая случайная траектория для C: [(0, 0, -0.1), (1, 0, -0.1), (1, 1, 1.0)]
2-ая случайная траектория для А: [(0, 0, -0.1), (1, 0, 1.0)]
2-ая случайная траектория для B: [(0, 2, 1.0)]
2-ая случайная траектория для C: [(0, 0, -0.1), (1, 1, 1.0)]
3-ая случайная траектория для А: [(0, 0, -0.1), (1, 0, 1.0)]
3-ая случайная траектория для B: [(0, 1, 0.5)]
3-ая случайная траектория для C: [(0, 1, -0.1), (2, 1, -0.1), (2, 1, -0.1), (2, 0, 1.0)]


### Return, expected return

Наиболее важная метрика оценки качества работы агента: отдача.

Отдача: $G(s_t) = \sum_{i=t}^T r_i$

Обычно также вводят параметр $\gamma \in [0, 1]$, дисконтирующий будущие вознаграждения. А еще, тк отдача может меняться от запуска к запуску благодаря вероятностным процессам, нас интересует отдача в среднем — ожидаемая отдача:

$$\hat{G}(s_t) = \mathbb{E} [ \sum_{i=t}^T \gamma^{i-t} r_i ]$$

Именно ее и оптимизируют в RL.

Давайте научимся считать отдачу для состояний по траектории и считать среднюю отдачу.

In [16]:
import numpy as np
from collections import defaultdict

def calculate_returns(trajectory, gamma=0.99):
    rewards = [step[2] for step in trajectory]
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

def evaluate_and_get_state_values(mdp, agent, n_episodes=1000, gamma=0.99):
    returns_per_state = defaultdict(list)
    for _ in range(n_episodes):
        trajectory = run_episode_term(mdp, agent)
        if not trajectory:
            continue
        returns = calculate_returns(trajectory, gamma=gamma)
        for t, step in enumerate(trajectory):
            state = step[0]
            G = returns[t]
            returns_per_state[state].append(G)
            
    avg_returns_per_state = {}
    all_states = mdp[0]
    for s in all_states:
        if s in returns_per_state:
            avg_returns_per_state[s] = np.mean(returns_per_state[s])
        else:
            avg_returns_per_state[s] = 0.0
            
    return avg_returns_per_state

In [17]:
optimal_agent_C = OptimalAgent(pi_C)

state_values_random = evaluate_and_get_state_values(C_mdp, random_agent_C)
state_values_optimal = evaluate_and_get_state_values(C_mdp, optimal_agent_C)

print("средняя отдача для C (gamma=0.99)\n")

print("--- случайный агент ---")
for state, value in sorted(state_values_random.items()):
    print(f"  G_hat(s{state}): {value:.3f}")

print("\n--- оптимальный агент ---")
for state, value in sorted(state_values_optimal.items()):
    print(f"  G_hat(s{state}): {value:.3f}")

средняя отдача для C (gamma=0.99)

--- случайный агент ---
  G_hat(s0): 0.733
  G_hat(s1): 0.769
  G_hat(s2): 0.799
  G_hat(s3): 0.000

--- оптимальный агент ---
  G_hat(s0): 0.890
  G_hat(s1): 1.000
  G_hat(s2): 0.000
  G_hat(s3): 0.000


In [18]:
optimal_agent_A = OptimalAgent(pi_A)

state_values_random = evaluate_and_get_state_values(A_mdp, random_agent_A)
state_values_optimal = evaluate_and_get_state_values(A_mdp, optimal_agent_A)

print("средняя отдача для A (gamma=0.99)\n")

print("--- случайный агент ---")
for state, value in sorted(state_values_random.items()):
    print(f"  G_hat(s{state}): {value:.3f}")

print("\n--- оптимальный агент ---")
for state, value in sorted(state_values_optimal.items()):
    print(f"  G_hat(s{state}): {value:.3f}")

средняя отдача для A (gamma=0.99)

--- случайный агент ---
  G_hat(s0): 0.890
  G_hat(s1): 1.000
  G_hat(s2): 0.000

--- оптимальный агент ---
  G_hat(s0): 0.890
  G_hat(s1): 1.000
  G_hat(s2): 0.000


In [19]:
optimal_agent_B = OptimalAgent(pi_B)

state_values_random = evaluate_and_get_state_values(B_mdp, random_agent_B)
state_values_optimal = evaluate_and_get_state_values(B_mdp, optimal_agent_B)

print("cредняя отдача для B (gamma=0.99)\n")

print("--- cлучайный агент ---")
for state, value in sorted(state_values_random.items()):
    print(f"  G_hat(s{state}): {value:.3f}")

print("\n--- оптимальный агент ---")
for state, value in sorted(state_values_optimal.items()):
    print(f"  G_hat(s{state}): {value:.3f}")

cредняя отдача для B (gamma=0.99)

--- cлучайный агент ---
  G_hat(s0): 0.532
  G_hat(s1): 0.000
  G_hat(s2): 0.000
  G_hat(s3): 0.000

--- оптимальный агент ---
  G_hat(s0): 1.000
  G_hat(s1): 0.000
  G_hat(s2): 0.000
  G_hat(s3): 0.000
